Verify the uploaded file

In [0]:
raw_path = "/Volumes/workspace/default/logisense_raw/"

files = dbutils.fs.ls(raw_path)

for file in files:
    print(file.name, file.size, "bytes")

archive.zip 44717580 bytes


Extract the ZIP

In [0]:

import zipfile
import os

zip_path = "/Volumes/workspace/default/logisense_raw/archive.zip"
extract_path = "/Volumes/workspace/default/logisense_raw/olist/"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed successfully!")


Extraction completed successfully!


Build the Bronze layer

1 — Verify extracted files

In [0]:

import os

olist_path = "/Volumes/workspace/default/logisense_raw/olist/"

csv_files = [
    f for f in os.listdir(olist_path)
    if f.endswith(".csv")
]

print("Number of CSV files:", len(csv_files))

for file in sorted(csv_files):
    print(file)


Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


2 — Create the Bronze ingestion code

In [0]:

from pyspark.sql import functions as F

source_path = "/Volumes/workspace/default/logisense_raw/olist/"

datasets = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

for table_name, file_name in datasets.items():

    file_path = source_path + file_name

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.default.bronze_{table_name}")
    )

    print(f"Created bronze_{table_name}: {df.count()} records")

print("BRONZE INGESTION COMPLETED!")


Created bronze_customers: 99441 records
Created bronze_orders: 99441 records
Created bronze_order_items: 112650 records
Created bronze_payments: 103886 records
Created bronze_reviews: 104162 records
Created bronze_products: 32951 records
Created bronze_sellers: 3095 records
Created bronze_geolocation: 1000163 records
Created bronze_category_translation: 71 records
BRONZE INGESTION COMPLETED!


3 — Verify our Bronze tables

In [0]:

tables = spark.sql("SHOW TABLES IN workspace.default")

bronze_tables = tables.filter(
    F.col("tableName").startswith("bronze_")
)

display(bronze_tables)


database,tableName,isTemporary
default,bronze_category_translation,false
default,bronze_customers,false
default,bronze_geolocation,false
default,bronze_order_items,false
default,bronze_orders,false
default,bronze_payments,false
default,bronze_products,false
default,bronze_reviews,false
default,bronze_sellers,false


In [0]:

display(
    spark.sql("""
        SELECT
            'customers' AS dataset,
            COUNT(*) AS total_records
        FROM workspace.default.bronze_customers

        UNION ALL

        SELECT 'orders', COUNT(*)
        FROM workspace.default.bronze_orders

        UNION ALL

        SELECT 'order_items', COUNT(*)
        FROM workspace.default.bronze_order_items

        UNION ALL

        SELECT 'reviews', COUNT(*)
        FROM workspace.default.bronze_reviews
    """)
)


dataset,total_records
customers,99441
orders,99441
order_items,112650
reviews,104162


inspect the orders table

In [0]:

display(
    spark.sql("""
        SELECT *
        FROM workspace.default.bronze_orders
        LIMIT 10
    """)
)


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z


In [0]:

display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_reviews,
            COUNT(DISTINCT review_id) AS unique_review_ids,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS missing_review_ids,
            SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS missing_order_ids
        FROM workspace.default.bronze_reviews
    """)
)


total_reviews,unique_review_ids,unique_order_ids,missing_review_ids,missing_order_ids
104162,102957,99742,1,2236


In [0]:
spark.table("workspace.default.bronze_orders").printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



Inspect order statuses

In [0]:

display(
    spark.sql("""
        SELECT
            order_status,
            COUNT(*) AS total_orders
        FROM workspace.default.bronze_orders
        GROUP BY order_status
        ORDER BY total_orders DESC
    """)
)


order_status,total_orders
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2
